# Pipeline benchmark (fixed CV)

Compare **6** tuned pipelines on **train** data with frozen hyperparameters from `data/processed/tuned/<model_id>.json`.

| Pipeline | Description |
|----------|-------------|
| `mspc_lr` | MSPC + elastic-net logistic |
| `mspc_rf` | MSPC + Random Forest |
| `xgb_mspc` | MSPC + XGBoost |
| `rf_k_lr` | RF top-K + elastic-net logistic |
| `rf_k_rf` | RF top-K + Random Forest |
| `rf_k_knn` | RF top-K + KNN |

Run GridSearchCV first: each notebook in `tuning/` (e.g. `tuning/mspc_lr.ipynb`).

**Primary metric: PR AUC (CV).** Hyperparameters tuned for mean PR AUC; decision threshold tuned to minimize BER. Leaderboards sorted by `mean_pr_auc`; holdout by `pr_auc`. Classification metrics (BER, TPR, TNR) use the tuned threshold.

**CV:** repeated stratified 5×5 on the 80% train split. **Holdout:** 20% test (reporting only).

In [1]:
import importlib
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload after editing scripts/ (avoids stale pipeline modules in kernel).
import scripts.benchmark_models as _bm
import scripts.mspc_features as _mf
import scripts.secom_pipelines as _sp

importlib.reload(_mf)
importlib.reload(_sp)
importlib.reload(_bm)

from scripts.benchmark_models import (
    BENCHMARK_RESULTS_PATH,
    build_benchmark_pipelines,
    run_holdout_benchmark,
    run_pipeline_benchmark,
    save_benchmark_results,
)
from scripts.secom_utils import load_all_tuned_params
from scripts.secom_pipelines import (
    TARGET_COL,
    feature_columns,
    load_mart,
    split_train_test,
)



In [2]:
tuned = load_all_tuned_params()
for model_id, payload in tuned.items():
    summary = payload.get("cv_summary", {})
    pr = summary.get("mean_pr_auc", "n/a")
    thr = payload.get("classifier_threshold", "n/a")
    print(
        model_id,
        f"mean_pr_auc={pr}",
        f"threshold={thr}",
        payload.get("grid_search_best_params", {}),
    )

mspc_lr mean_pr_auc=0.189514214859201 threshold=0.35000000000000003 {'preprocess__sensor_mspc__pls__n_components': 10, 'classifier__C': 0.01, 'classifier__l1_ratio': 0.95}
mspc_rf mean_pr_auc=0.18914013506581448 threshold=0.25 {'preprocess__sensor_mspc__pls__n_components': 15, 'classifier__max_depth': 6}
xgb_mspc mean_pr_auc=0.1818675778753093 threshold=0.05 {'preprocess__sensor_mspc__pls__n_components': 25, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.03}
rf_k_lr mean_pr_auc=0.18249764714885658 threshold=0.5 {'preprocess__sensor_mspc__select__max_features': 40, 'classifier__C': 1.0, 'classifier__l1_ratio': 0.95}
rf_k_rf mean_pr_auc=0.19898247336029584 threshold=0.2 {'preprocess__sensor_mspc__select__max_features': 50, 'classifier__max_depth': 8}
rf_k_knn mean_pr_auc=0.16972901572495272 threshold=0.05 {'preprocess__sensor_mspc__select__max_features': 40, 'classifier__n_neighbors': 10}


In [3]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].astype(int)
print(len(X_train), len(X_test), y_train.mean())

1253 314 0.06624102154828412


In [4]:
pipelines = build_benchmark_pipelines(tuned)
list(pipelines.keys())

['mspc_lr', 'mspc_rf', 'xgb_mspc', 'rf_k_lr', 'rf_k_rf', 'rf_k_knn']

In [5]:
leaderboard = run_pipeline_benchmark(pipelines, X_train, y_train, show_progress=True)
display(leaderboard)

/home/troy/SECOM/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/troy/SECOM/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


RuntimeError: MSPC preprocess did not emit hotelling_t2 and q_statistic.

In [ ]:
holdout = run_holdout_benchmark(
    pipelines, X_train, y_train, X_test, y_test, show_progress=True
)
display(holdout)

Holdout: 6 pipelines (reporting only)
  mspc_lr: holdout PR AUC 0.149, BER 40.4%
  mspc_rf: holdout PR AUC 0.142, BER 27.9%
  xgb_mspc: holdout PR AUC 0.134, BER 46.4%
  rf_k_lr: holdout PR AUC 0.144, BER 37.1%
  rf_k_rf: holdout PR AUC 0.233, BER 24.9%
  rf_k_knn: holdout PR AUC 0.113, BER 40.4%


,pipeline,true_positive_rate,true_negative_rate,balanced_accuracy,ber_percent,true_positive_percent,true_negative_percent,confusion_matrix,roc_auc,pr_auc
0,rf_k_rf,0.857143,0.645051,0.751097,24.890297,85.714286,64.505119,"[[189, 104], [3, 18]]",0.786446,0.233432
1,mspc_lr,0.619048,0.573379,0.596213,40.378677,61.904762,57.337884,"[[168, 125], [8, 13]]",0.677068,0.148519
2,rf_k_lr,0.476190,0.781570,0.628880,37.111978,47.619048,78.156997,"[[229, 64], [11, 10]]",0.665204,0.143752
3,mspc_rf,0.714286,0.726962,0.720624,27.937591,71.428571,72.696246,"[[213, 80], [6, 15]]",0.725825,0.141644
4,xgb_mspc,0.190476,0.880546,0.535511,46.448887,19.047619,88.054608,"[[258, 35], [17, 4]]",0.673655,0.133965
5,rf_k_knn,0.523810,0.668942,0.596376,40.362425,52.380952,66.894198,"[[196, 97], [10, 11]]",0.625874,0.112762


In [ ]:
comparison = (
    leaderboard.merge(
        holdout,
        on="pipeline",
        suffixes=("_cv", "_holdout"),
    )[
        [
            "pipeline",
            "mean_pr_auc",
            "pr_auc",
            "mean_roc_auc",
            "roc_auc",
            "mean_ber_percent",
            "ber_percent",
        ]
    ]
    .rename(
        columns={
            "mean_pr_auc": "pr_auc_cv",
            "pr_auc": "pr_auc_holdout",
            "mean_roc_auc": "roc_auc_cv",
            "roc_auc": "roc_auc_holdout",
            "mean_ber_percent": "ber_cv",
            "ber_percent": "ber_holdout",
        }
    )
    .sort_values("pr_auc_cv", ascending=False)
)
display(comparison)

,pipeline,pr_auc_cv,pr_auc_holdout,roc_auc_cv,roc_auc_holdout,ber_cv,ber_holdout
0,mspc_rf,0.201979,0.141644,0.702085,0.725825,33.739442,27.937591
1,mspc_lr,0.197639,0.148519,0.700197,0.677068,33.961287,40.378677
2,rf_k_rf,0.196092,0.233432,0.723416,0.786446,34.043426,24.890297
3,xgb_mspc,0.190413,0.133965,0.691239,0.673655,37.234540,46.448887
4,rf_k_lr,0.173357,0.143752,0.685131,0.665204,37.383924,37.111978
5,rf_k_knn,0.156187,0.112762,0.665134,0.625874,35.642722,40.362425


In [ ]:
save_benchmark_results(
    tuned,
    leaderboard,
    holdout,
    train_rows=len(train_df),
    test_rows=len(test_df),
)
print(f"Wrote {BENCHMARK_RESULTS_PATH}")

Wrote /home/troy/SECOM/data/processed/secom_pipeline_benchmark.json


## Top features: `rf_k_rf`

Fit on train and rank **RandomForest** `feature_importances_` against preprocess output names (selected sensors, `hotelling_t2`, passthrough aux).

In [ ]:
import pandas as pd

from scripts.secom_utils import fitted_base_classifier

PIPELINE_NAME = "rf_k_rf"
rf_k30 = pipelines[PIPELINE_NAME]
rf_k30.fit(X_train, y_train)

feature_names = rf_k30.named_steps["preprocess"].get_feature_names_out()
importances = fitted_base_classifier(rf_k30).feature_importances_

top_features = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
top_features["importance_pct"] = 100 * top_features["importance"] / top_features["importance"].sum()

print(f"{PIPELINE_NAME}: top {len(top_features)} features by RF importance (train fit)")
display(top_features.head(50))

rf_k_rf: top 86 features by RF importance (train fit)


,feature,importance,importance_pct
0,c_103,0.052273,5.227323
1,hotelling_t2,0.045028,4.502850
2,c_59,0.044229,4.422910
3,c_33,0.038727,3.872715
4,c_477,0.029433,2.943284
5,c_31,0.025604,2.560357
6,c_510,0.024233,2.423281
7,c_183,0.023642,2.364170
8,c_130,0.023609,2.360888
9,c_351,0.023420,2.342038


## Top features: `rf_k_lr`

Fit on train and rank **elastic-net logistic** coefficients (`|coef_|`) against preprocess output names (K=30 selected sensors, `hotelling_t2`, passthrough aux).

In [ ]:
import numpy as np

from scripts.secom_utils import fitted_base_classifier

PIPELINE_NAME = "mspc_lr"
lr_k30 = pipelines[PIPELINE_NAME]
lr_k30.fit(X_train, y_train)

feature_names = lr_k30.named_steps["preprocess"].get_feature_names_out()
coefs = fitted_base_classifier(lr_k30).coef_.ravel()

top_features_lr = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": coefs,
            "abs_coefficient": np.abs(coefs),
        }
    )
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)
top_features_lr["abs_coef_pct"] = (
    100 * top_features_lr["abs_coefficient"] / top_features_lr["abs_coefficient"].sum()
)

n_nonzero = int((top_features_lr["coefficient"] != 0).sum())
print(
    f"{PIPELINE_NAME}: {n_nonzero} / {len(top_features_lr)} non-zero coefs (train fit)"
)
display(top_features_lr.head(50))

mspc_lr: 22 / 52 non-zero coefs (train fit)


,feature,coefficient,abs_coefficient,abs_coef_pct
0,pls_1,-1.173490,1.173490,13.478113
1,pls_0,1.101274,1.101274,12.648681
2,pls_4,0.944424,0.944424,10.847174
3,pls_2,-0.879395,0.879395,10.100287
4,pls_6,-0.616877,0.616877,7.085137
5,pls_5,0.588632,0.588632,6.760726
6,pls_3,-0.556625,0.556625,6.393119
7,pls_12,-0.507340,0.507340,5.827048
8,pls_7,0.325234,0.325234,3.735480
9,pls_10,-0.267235,0.267235,3.069327
